In [1]:
# Calculate PM25 population weighted exposure

In [2]:
import os
import xarray as xr
import numpy as np
from utils.utils import get_scenario_config

In [3]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

In [4]:
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASK_DIR, mask_file)
country_mask = xr.open_dataarray(mask_path)

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
pop_ssp2 = xr.open_dataarray(pop_path)
pop = pop_ssp2.reindex_like(country_mask, method="nearest", tolerance=1e-9)

In [5]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

PM_DIR = f"/glade/work/awells/air_quality/{model}/pm25/annual_pm25_bc/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/exposure/"

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")

    dates = dates = f"{years.start}-{years.stop}"

    pm25_file = f"Annual_PM25_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    pm25_path = os.path.join(PM_DIR, pm25_file)
    pm25 = xr.open_dataarray(pm25_path)
    # Select same years as pm25 data for population
    population = pop.sel(year=pm25.year)

    # Adjust indices to match (with small tolerance)
    # e.g., max 1e-7 km distance
    pm25 = pm25.reindex_like(country_mask, method="nearest",
                             tolerance=1e-9, fill_value=0)

    weighted_value = population * pm25

    country_list = []

    for country in country_mask.country:
        print(f"Processing country number {country.data}")
        mask = country_mask.sel(country=country)
        country_weight = xr.where(
            mask == 1,
            weighted_value,
            np.nan).sum(dim=("lat", "lon"))
        pop_country = xr.where(
            mask == 1,
            population,
            np.nan).sum(dim=("lat", "lon"))
        country_pop_weighted = country_weight / pop_country
        country_list.append(country_pop_weighted)

    pop_weighted_exposure = xr.concat(country_list, "country")

    description = ("Annual mean PM2.5 population weighted exposure by country "
                   "- scripts by A.F. Wells (2025)")
    pop_weighted_exposure.attrs["description"] = description
    pop_weighted_exposure.attrs["ensemble_number"] = ens_num
    pop_weighted_exposure.attrs["scenario"] = scenario
    pop_weighted_exposure.attrs["model"] = model

    out_file = f"Annual_PM25_country_population_weighted_exposure_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving to {out_path}")
    pop_weighted_exposure.to_netcdf(out_path)

print("All processing complete.")

Processing G6-1.5K, Ensemble 01
Processing country number Armenia
Processing country number Azerbaijan
Processing country number Georgia
Processing country number Kazakhstan
Processing country number Kyrgyzstan
Processing country number Mongolia
Processing country number Tajikistan
Processing country number Turkmenistan
Processing country number Uzbekistan
Processing country number Albania
Processing country number Bosnia and Herzegovina
Processing country number Bulgaria
Processing country number Croatia
Processing country number Czech Republic
Processing country number Hungary
Processing country number Macedonia
Processing country number Montenegro
Processing country number Poland
Processing country number Romania
Processing country number Serbia
Processing country number Slovakia
Processing country number Slovenia
Processing country number Belarus
Processing country number Estonia
Processing country number Latvia
Processing country number Lithuania
Processing country number Moldova
